In [4]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms import community
import plotly.graph_objects as go

In [5]:
edges_path = 'edges.csv'
hero_network_path = 'hero-network.csv'
nodes_path = 'nodes.csv'

edges_df = pd.read_csv(edges_path)
hero_network_df = pd.read_csv(hero_network_path)
nodes_df = pd.read_csv(nodes_path)

# Create a dictionary that maps each hero to a set of heroes they appear with
hero_pairs = {}

for index, row in hero_network_df.iterrows():
    hero1 = row['hero1']
    hero2 = row['hero2']
    
    if hero1 in hero_pairs:
        hero_pairs[hero1].add(hero2)
    else:
        hero_pairs[hero1] = set([hero2])
    
    if hero2 in hero_pairs:
        hero_pairs[hero2].add(hero1)
    else:
        hero_pairs[hero2] = set([hero1])

# Convert the dictionary into a list of tuples for easier display
hero_pairs_list = [(hero, list(paired_heroes)) for hero, paired_heroes in hero_pairs.items()]

# Create a DataFrame from the hero pairs list
hero_pairs_df = pd.DataFrame(hero_pairs_list, columns=['Hero', 'Appeared With'])

hero_pairs_df['Appeared With'] = hero_pairs_df['Appeared With'].apply(lambda x: ', '.join(x))


In [6]:
hero_pairs_df.to_csv("hero_pairs.csv", index=False)
hero_pairs_df.head(10)

,Hero,Appeared With
0,"LITTLE, ABNER","PRINCESS ZANDA, BLACK PANTHER/T'CHAL, CARNIVOR..."
1,PRINCESS ZANDA,"BLACK PANTHER/T'CHAL, LITTLE, ABNER, CARNIVORE..."
2,BLACK PANTHER/T'CHAL,"PRINCESS ZANDA, NIGHT THRASHER/DUANE, KNIGHT E..."
3,"STEELE, SIMON/WOLFGA","ERWIN, CLYTEMNESTRA, IRON MAN/TONY STARK , RAV..."
4,"FORTUNE, DOMINIC","ERWIN, CLYTEMNESTRA, IRON MAN/TONY STARK , RAV..."
5,"ERWIN, CLYTEMNESTRA","CABE, BETHANY, SCORPIO II, IRON MAN/TONY STARK..."
6,IRON MAN/TONY STARK,"RADIOACTIVE MAN/DR. , ZARRKO, ARTHUR, DR. OCTO..."
7,IRON MAN IV/JAMES R.,"ZIMMER, ABE, CABE, BETHANY, RADIOACTIVE MAN/DR..."
8,"RAVEN, SABBATH II/EL","ERWIN, CLYTEMNESTRA, IRON MAN/TONY STARK , FOR..."
9,CARNIVORE/COUNT ANDR,"PRINCESS ZANDA, LITTLE, ABNER, BLACK PANTHER/T..."


In [8]:
G = nx.Graph()

# Add edges from the hero_network_df
for index, row in hero_network_df.iterrows():
    G.add_edge(row['hero1'], row['hero2'])

# Centrality analysis: Compute the degree centrality of the nodes
degree_centrality = nx.degree_centrality(G)

# Community detection: Using the Louvain method for community detection
communities = community.greedy_modularity_communities(G)
community_map = {}
for i, com in enumerate(communities):
    for node in com:
        community_map[node] = i

# Assign a community ID to each node for visualization
nx.set_node_attributes(G, community_map, 'community')

pos = nx.spring_layout(G, k=0.05, iterations=20)

# Calculate degree centrality
degree_centrality = nx.degree_centrality(G)

# Identify the top 10 heroes with the highest degree centrality
top_10_heroes = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]

# Display top 10 heroes and the number of detected communities
(top_10_heroes, len(communities))

([('CAPTAIN AMERICA', 0.2526002971768202),
  ('SPIDER-MAN/PETER PAR', 0.24219910846953938),
  ('IRON MAN/TONY STARK ', 0.23476968796433878),
  ('BEAST/HENRY &HANK& P', 0.18722139673105498),
  ('ANGEL/WARREN KENNETH', 0.17682020802377413),
  ('CYCLOPS/SCOTT SUMMER', 0.17682020802377413),
  ('HULK/DR. ROBERT BRUC', 0.16047548291233282),
  ('MARVEL GIRL/JEAN GRE', 0.1575037147102526),
  ('THING/BENJAMIN J. GR', 0.1515601783060921),
  ('SCARLET WITCH/WANDA ', 0.14561664190193163)],
 17)

In [9]:
# Extract node and edge positions from the layout
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)  # None values are used by Plotly to separate lines
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

# Create a trace for the edges
edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

# Create a trace for the nodes
node_x = []
node_y = []
text = []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    text.append(node)  # The node's text label

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    text=text,
    marker=dict(
        showscale=True,
        # Color scale options: 'Greys', 'YlGnBu', 'Greens', 'YlOrRd', 'Bluered', 'RdBu',
        # 'Reds', 'Blues', 'Picnic', 'Rainbow', 'Portland', 'Jet', 'Hot', 'Blackbody', 'Earth',
        # 'Electric', 'Viridis', 'Cividis'
        colorscale='YlGnBu',
        reversescale=True,
        color=[],
        size=10,
        colorbar=dict(
            thickness=15,
            title='Node Connections',
            xanchor='left',
            titleside='right'
        ),
        line_width=2))

# Color node points by the number of connections
node_adjacencies = []
for node, adjacencies in enumerate(G.adjacency()):
    node_adjacencies.append(len(adjacencies[1]))
node_trace.marker.color = node_adjacencies

# Create a figure
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title='<br>Hero Network with Community Detection and Central Nodes Highlighted',
                titlefont_size=16,
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                annotations=[ dict(
                    text="NetworkX and Plotly",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002 ) ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )

fig.show()